# Experiment 7 - Texture Analysis using GLCM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.feature import graycomatrix, graycoprops

In [ ]:
LEVELS = 16

textures = {
    "brick": data.brick(),
    "grass": data.grass(),
    "gravel": data.gravel(),
}

def quantize(src, levels=LEVELS):
    return (src.astype(np.int32) * levels // 256).astype(np.uint8)

def glcm(q, dy, dx, levels=LEVELS):
    h, w = q.shape
    a = q[max(0, -dy):h - max(0, dy), max(0, -dx):w - max(0, dx)]
    b = q[max(0, dy):h - max(0, -dy), max(0, dx):w - max(0, -dx)]
    m = np.zeros((levels, levels), dtype=np.float64)
    np.add.at(m, (a.ravel(), b.ravel()), 1)
    m = m + m.T
    return m / m.sum()

def features(p):
    i, j = np.indices(p.shape)
    mi, mj = (i * p).sum(), (j * p).sum()
    si = np.sqrt((((i - mi) ** 2) * p).sum())
    sj = np.sqrt((((j - mj) ** 2) * p).sum())
    return {
        "contrast": ((i - j) ** 2 * p).sum(),
        "dissimilarity": (np.abs(i - j) * p).sum(),
        "homogeneity": (p / (1 + (i - j) ** 2)).sum(),
        "energy": np.sqrt((p ** 2).sum()),
        "correlation": (((i - mi) * (j - mj) * p).sum()) / (si * sj),
    }

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for col, (name, im) in enumerate(textures.items()):
    axes[0, col].imshow(im, cmap="gray")
    axes[0, col].set_title(name)
    axes[0, col].axis("off")
    axes[1, col].imshow(glcm(quantize(im), 0, 1), cmap="magma")
    axes[1, col].set_title(f"GLCM d=1, 0 deg ({LEVELS} levels)")
plt.tight_layout()
plt.show()

In [ ]:
print(f"{'texture':8s} " + " ".join(f"{k:>13s}" for k in features(np.eye(2) / 2)))
for name, im in textures.items():
    f = features(glcm(quantize(im), 0, 1))
    print(f"{name:8s} " + " ".join(f"{v:13.4f}" for v in f.values()))

In [ ]:
q = quantize(textures["brick"])
lib = graycomatrix(q, [1], [0], levels=LEVELS, symmetric=True, normed=True)
mine = features(glcm(q, 0, 1))
for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation"]:
    print(f"{prop:13s} own={mine[prop]:.5f}  skimage={graycoprops(lib, prop)[0, 0]:.5f}")

In [ ]:
offsets = {"0": (0, 1), "45": (-1, 1), "90": (-1, 0), "135": (-1, -1)}
dists = [1, 2, 4, 8]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, im in textures.items():
    qi = quantize(im)
    axes[0].plot(dists, [features(glcm(qi, 0, d))["contrast"] for d in dists], marker="o", label=name)
    axes[1].bar([f"{name}\n{a}" for a in offsets], [features(glcm(qi, *o))["homogeneity"] for o in offsets.values()])
axes[0].set_xlabel("distance (px)")
axes[0].set_ylabel("contrast")
axes[0].set_title("contrast vs distance, 0 deg")
axes[0].legend()
axes[1].set_title("homogeneity by angle, d=1")
axes[1].tick_params(axis="x", labelsize=7)
plt.tight_layout()
plt.show()